# Denoising, Motion Correction, and ROI

Created based on scripts from Portugues lab and CSHL2024 TAs, commented by Adam S. Charles

## Loading packages and functions
Understand what the packages below are and do.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from scipy.stats import zscore
import tifffile
from scipy.spatial.distance import euclidean

If you see `ModuleNotFoundError: No module named 'module_name'`, run the following command in a cell  
`!pip install module_name`

In [ ]:
# put the path to your folder holding the calcium imaging recording and the DeepLabCut csv
# path = Path(r"/home/estherwhang/cshl2025/Data/Somatic-big/file_00001.tif") # you need the `r` before "" only on windows

## 2-photon dataset
Load the tiff file  of your video as a numpy array `frames`.  
It is a three dimensional array (T, Y, X) imaged from the same plane at different times. You can also open the tiff with ImageJ to see it as a movie.

In [ ]:
frames = tifffile.imread("/home/estherwhang/cshl2025/Data/Somatic-big/file_00001.tif")

Use `np.shape` to query the dimensions of the array `frames`. Does this make sense from what you observed in ImageJ?

In [ ]:
print(frames.shape)

Generate an anatomy image called `original_anatomy` of this plane by averaging all the frames.  

In [ ]:
frame_rate_exp = 15
time_vector = np.linspace(0,frames.shape[0]/15,frames.shape[0])

In [ ]:
original_anatomy = frames.mean(0)

In [ ]:
fig = plt.figure(figsize=(4, 4))
ax = fig.add_subplot()
ax.imshow(
    original_anatomy,
    cmap="gray",
    vmin=np.percentile(original_anatomy, 2.5),
    vmax=np.percentile(original_anatomy, 97.5),
)
plt.show()

In [ ]:
# Lets try some denoising! 

# Note that this portion will take some time
# on the server it took ~5 minutes for a (2000, 256, 256) video

from scipy import ndimage

gaussWidth = 3
filterSize = 2*gaussWidth
pix4filt   = np.arange(-filterSize,filterSize)**2

# Temporal filters operate in 1D, so we need a 1D Gaussian bump:
gFilt1D = np.exp(-(pix4filt)/(2*(gaussWidth)**2))
gFilt1D = gFilt1D/gFilt1D.sum()
gFilt1D = np.expand_dims(np.expand_dims(gFilt1D,axis=1),axis=2)
clearFrames1D = ndimage.convolve(frames,gFilt1D)
clearFrames1D = np.array(clearFrames1D)

# Next we'll use a 2D filter.
# Spatial filters operate in 2D, so we need a 2D Gaussian bump:
gFilt2D = np.exp(-(pix4filt[:,np.newaxis] + pix4filt)/(2*(gaussWidth)**2))
gFilt2D = gFilt2D/gFilt2D.sum(axis=None)
gFilt2D = np.expand_dims(gFilt2D,axis=0)
clearFrames2D = ndimage.convolve(frames,gFilt2D)
clearFrames2D = np.array(clearFrames2D)

## Space-time blurring needs a 3D bump!
gFilt3D = np.exp(-(pix4filt[np.newaxis,np.newaxis,:] + pix4filt[np.newaxis,:,np.newaxis] + pix4filt[:,np.newaxis,np.newaxis])/(2*(gaussWidth)**2))
gFilt3D = gFilt3D/gFilt3D.sum(axis=None)
clearFrames3D = ndimage.convolve(frames,gFilt3D)
clearFrames3D = np.array(clearFrames3D)


In [ ]:
## Finally here's an easier "PCA denoising"
#from scipy import sparse, linalg, stats
from scipy.sparse.linalg import svds

movieSize = frames.shape
frames2 = np.array(np.reshape(frames, (movieSize[0],movieSize[1]*movieSize[2])))
frames2.shape
U, S, Vt = svds(np.array(frames2).astype(float), k=50)
clearFramesPCA = U @ np.diag(S) @ Vt
frames = np.reshape(frames, movieSize)
clearFramesPCA = np.reshape(clearFramesPCA, movieSize)

In [ ]:

import plotly.express as px
import plotly.io as pio

bothMovies = np.concatenate((np.concatenate((clearFrames1D[0:100,:,:], clearFrames2D[0:100,:,:]), axis=2),
                             np.concatenate((clearFrames3D[0:100,:,:], clearFramesPCA[0:100,:,:]), axis=2)),1)

fig = px.imshow(bothMovies[0:100,:,:], zmin=0, zmax=1000, animation_frame=0, binary_string=True, labels=dict(animation_frame="slice"))
fig.show()
#np.ravel(clearFrames).min()
#pio.renderers
#fig

Now we will use the function `phase_cross_correlation` from `skimage`.  
We want to perform sub-pixel corrections at 0.1 of a pixel, so use the `upsample_factor` argument. Save the shifts in the X and Y directions in two arrays called `x_shift` and `y_shift`, and also compute a `total_shift` array (use pythagoras).

In [ ]:
from skimage.registration import phase_cross_correlation

In [ ]:
x_shift = np.empty(len(frames))
y_shift = np.empty(len(frames))

for i in range(len(frames)):
    result, _, _ = phase_cross_correlation(original_anatomy, frames[i,:,:], upsample_factor=10)
    x_shift[i] = result[1]
    y_shift[i] = result[0]

total_shift = np.sqrt(x_shift ** 2 + y_shift ** 2)

Plot the `total_shift` of each frame. Do you think the motion arises from continuous drifts or acute motor events?  
In a second subplot, plot a scatter plot of the x- and y-shift at each frame. You can choose to add some jitter so you can visualize dots that are superimposed on one another.

In [ ]:
fig = plt.figure(figsize=(8, 4))

ax = fig.add_subplot(121)
ax.plot(time_vector,total_shift);
ax.set(xlabel="time (s)", ylabel="total shift")

ax = fig.add_subplot(122)
ax.scatter(
    x_shift + 0.04 * np.random.rand(len(x_shift)) - 0.02,
    y_shift + 0.04 * np.random.rand(len(x_shift)) - 0.02,
    s=0.5,
)
ax.set(xlabel="x shift", ylabel="y shift")
plt.tight_layout()
plt.show()

## Second order correction of motion artifact
Correct the individual frames by performing an individual sub-pixel alignment frame by frame. In order to do this use the function `shift` from `scipy.ndimage`.  
Use this corrected stack to generate a new anatomy image `aligned_anatomy`.

In [ ]:
from scipy.ndimage import shift

In [ ]:
aligned_frames =np.empty(frames.shape)
for i in range(len(frames)):
    aligned_frames[i,:,:] = shift(frames[i,:,:], (y_shift[i], x_shift[i]))

aligned_anatomy = aligned_frames.mean(0)

Compare the original and the aligned anatomy by visualizing pixel-wise difference between them

In [ ]:
diff = original_anatomy - aligned_anatomy

In [ ]:
fig = plt.figure(figsize=(12, 4))

ax = fig.add_subplot(131)
ax.imshow(
    original_anatomy,
    cmap="gray",
    vmin=None,
    vmax=None,
);
ax.set(title="original");

ax = fig.add_subplot(132)
ax.imshow(
    diff,
    vmin=np.percentile(diff, 2.5),
    vmax=np.percentile(diff, 97.5),cmap="coolwarm",
);
ax.set(title="difference")

ax = fig.add_subplot(133)
ax.imshow(
    aligned_anatomy,
    cmap="gray",
    vmin=None,
    vmax=None,
)
ax.set(title="aligned")
plt.show()

Align the ORIGINAL frames to this new aligned anatomy.  
Compute the total shift of each frame to this `aligned_anatomy` and plot this shift on top of the shift you obtained for the first alignment. Why are these different?

In [ ]:
#find the shift of the original frames with the better template
x_shift_2 = np.empty(len(frames))
y_shift_2 = np.empty(len(frames)) 

for i in range(len(frames)):
    result, _, _ = phase_cross_correlation(aligned_anatomy, frames[i,:,:], upsample_factor=10)
    x_shift_2[i] = result[1]
    y_shift_2[i] = result[0]

total_shift_2 = np.sqrt(x_shift_2 ** 2 + y_shift_2 ** 2)

In [ ]:
fig = plt.figure(figsize=(8, 4))

ax = fig.add_subplot()
ax.plot(time_vector,total_shift)
ax.plot(time_vector,total_shift_2)
ax.legend(["first", "second"])
ax.set(xlabel="time (s)",ylabel="total shift")
plt.show()

# we plot the new shift in orange
# What benefits were there to aligning to the second template?

Finally shift the original frames by this new amount to obtain a final stack `final_frames`, and average this stack to obtain `final_anatomy`.

In [ ]:
# Let's implement the shift!
final_frames =np.empty(frames.shape)
for i in range(frames.shape[0]):
    final_frames[i,:,:] = shift(frames[i,:,:], (y_shift_2[i], x_shift_2[i]))

final_anatomy = final_frames.mean(0)

In [ ]:
# Compare the original anatomy to the final anatomy
diff = original_anatomy - final_anatomy

In [ ]:
fig = plt.figure(figsize=(12, 8))

ax = fig.add_subplot(231)
ax.imshow(
    original_anatomy,
    cmap="gray",
    vmin=None,
    vmax=None,
)
ax.set(title="original")

ax = fig.add_subplot(232)
ax.imshow(
    diff,
    vmin=np.percentile(diff, 2.5),
    vmax=np.percentile(diff, 97.5),
    cmap="coolwarm",
)
ax.set(title="difference");

ax = fig.add_subplot(233)
ax.imshow(
    final_anatomy,
    cmap="gray",
    vmin=None,
    vmax=None,
)
ax.set(title="aligned")

ax = fig.add_subplot(234)
ax.imshow(original_anatomy[120:160, 160:210], cmap="gray")

ax = fig.add_subplot(235)
ax.imshow(diff[120:160, 160:210],cmap="coolwarm")

ax = fig.add_subplot(236)
ax.imshow(final_anatomy[120:160, 160:210], cmap="gray")
plt.show()

## Creating a correlation map
Use the function `pearsonr` from `scipy.stats` to compute the correlation between the fluorescence time series of one pixel and the fluorescence time series of its eight surrounding pixels summed. Ignore boundary pixels for now.  
Plot it alongside the anatomy.

In [ ]:
from scipy.stats import pearsonr #import the package that will allow you to calculate correlation

In [ ]:
correlation_map = np.zeros(final_anatomy.shape)                             # initialize an empty array
for y in range(1, correlation_map.shape[0]-1):                              # For every pixel in a frame
    for x in range(1, correlation_map.shape[1]-1):
        this_px = final_frames[:, y, x]                                     # select pixel and its time series over the length of the video
        surr_px = final_frames[:, y-1:y+2, x-1:x+2].sum((1,2)) - this_px    # get the sum of the eight surrounding pixels's time series
        r, _ = pearsonr(this_px, surr_px)                                   # find correlation to surrounding pixels
        correlation_map[y, x] = r                                           # put final correlation value to correlation map

# one note: this will take a while (length dependent on size of dataframe)
# For this demo, this cell took around one minute

In [ ]:
# Let's look at our resulting correlation map!
fig = plt.figure(figsize=(8, 4))

ax = fig.add_subplot(121)
ax.imshow(
    final_anatomy,
    vmin=None,
    vmax=None,cmap="gray")
ax.set(title="anatomy")

ax = fig.add_subplot(122)
im = ax.imshow(
    correlation_map,
    vmin=0,
    vmax=1,cmap="gray")
cbar = fig.colorbar(im, ax=ax, orientation='vertical')
cbar.set_label('Correlation')
ax.set(title="correlation map")
plt.show()

# compare the anatomy to the correlation map
# Admire how well the correlation map works!

Plot a histogram of the correlation values.

In [ ]:
# Let's plot a histogram to see what we have: 
#  
plt.figure()
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot()
ax.hist(correlation_map.flatten(), bins=300);
plt.show()

# What is the distribution of our correlation map?

## ROI extraction

We will now generate a function that extracts ROIs (regions of interest).  
The function should take 4 inputs: a correlation map, the final stack of aligned frames, a correlation threshold value, and a maximum size of an ROI in pixels.

**The function should:**
- Identify the pixel with the highest correlation value in the correlation map (the ROI seed). Initialize the list of pixels in this ROI with this seed.
- Compute the individual correlations of the fluorescence time-series in this pixel with its 8 neighbours. A convenient way of idetifying the neighbours of a region is by dilating the reion by one pixel. You can do this with `binary_dilation` from `skimage.morphology`.
- Add the neighbours whose correlation exceeds the threshold to the ROI.
- Repeat by looking at the neighbouring pixels of the new ROI and computing the correlation of their fluorescence with the total fluorescence of the ROI. Add the pixels to the ROI if they exceed the threshold.
- Stop the function if no neighbouring pixels exceed the threshold or if the size of the ROI exceeds the maximum size (the last input to the function).
- The function should return the pixels within the ROI, the fluoresence trace of the ROI, the size of the ROI and the correlation map with the values of the pixels in the extracted ROI set to zero.

In [ ]:
from skimage.morphology import binary_dilation

In [ ]:
# this function finds the ROI based on the correlation map we generated in the cells above

# the function works by starting at the highest points of the correlation map,
# then slowly adding surrounding pixels to "grow" out an ROI


def next_roi(corr_map, frames, corr_threshold, size_threshold):
    i, j = np.unravel_index(np.argmax(corr_map), corr_map.shape)
    this_trace = frames[:, i, j]
    this_roi = np.zeros(corr_map.shape, dtype=np.uint8)
    this_roi[i, j] = 1
    this_corr_map = corr_map.copy()
    this_corr_map[i, j] = 0

    growing = True
    while growing and (this_roi.sum() < size_threshold):
        growing = False

        dilated = binary_dilation(this_roi, np.ones((3, 3)))
        neighbours = np.argwhere(dilated - this_roi)

        trace_to_be_added = np.zeros(len(this_trace))

        for n in range(neighbours.shape[0]):
            i, j = neighbours[n, :]
            if this_corr_map[i, j] != 0:
                px_trace = frames[:, i, j]
                r, _ = pearsonr(this_trace, px_trace)
                
                if r > corr_threshold: # if the correlation between the current pixel trace and neighboring pixel trace is over our threshold
                    growing = True # we grow out the ROI
                    this_roi[i, j] = 1
                    this_corr_map[i, j] = 0
                    trace_to_be_added += px_trace
        this_trace += trace_to_be_added

    return this_roi, this_trace, this_roi.sum(), this_corr_map

Use the function you have written to extract 300 ROIs within the plane. Use a correlation threshold value of 0.3 and a maximum ROI size of 200 pixels.  
Plot a figure with four subplots:
- The first one should be an image of the fluorescence time series of the ROIs. Make sure that you plot the normalized fluorescence for each ROI to be able to compare them. Import `zscore` from `scipy.stats` to do this.  
- The second one should be a map showing the ROIs extracted.  
- The third one should be the original correlation map with the correlation of the extracted ROIs set to zero.  
- The fourth one should be the original correlation map.

In [ ]:
correlation_map[:5, :] = 0
correlation_map[:, :5] = 0
correlation_map[-5:, :] = 0
correlation_map[:, -5:] = 0

In [ ]:

# make sure the following parameters are updated for your particular video: 
# for example, a 512 by 512 video will have a much larger area for pixels,
# which means you should increase size_range or size_threshold

# Using something like ImageJ or plt.imshow of one of the frames is a good quick way of guessing the size

size_range = [15,100]
n_rois = 100
corr_threshold = 0.3
size_threshold = 200

traces_raw = np.empty((n_rois, len(final_frames)))*np.nan
roi_map = np.zeros(final_anatomy.shape, dtype=np.uint16)

used_corr_map = correlation_map.copy()

for i in tqdm(range(n_rois)):
    this_roi, this_trace, this_size, used_corr_map = next_roi(
        used_corr_map,
        final_frames,
        corr_threshold,
        size_threshold,
    )
    if (this_size>size_range[0])and(this_size<size_range[-1]): # size threshold
        traces_raw[i, :] = this_trace
        roi_map += this_roi * (i+1)
traces_raw = traces_raw[np.sum(np.isfinite(traces_raw),1)>0,:]

In [ ]:
traces = zscore(traces_raw, axis=1)

In [ ]:
fig = plt.figure(figsize=(9, 6), tight_layout=True)

ax = fig.add_subplot(211)
img = ax.imshow(traces[np.argsort(np.argmax(traces,1)),:], cmap="afmhot", vmin=0, vmax=np.percentile(traces, 99),
               extent=[time_vector[0],time_vector[-1],1,traces.shape[0]+1])
ax.set(aspect="auto", xlabel="time (s)", ylabel="ROI #")
cbar = fig.colorbar(img, ax=ax)
cbar.set_label('z-scored F')

ax = fig.add_subplot(234)
ax.imshow(roi_map)

ax = fig.add_subplot(235)
ax.imshow(used_corr_map)

ax = fig.add_subplot(236)
ax.imshow(correlation_map)
plt.show()

Save the zscored traces as a numpy data file, and the ROI map as a tiff file.

In [ ]:
username = None #Fill in your username
# username = "estherwhang" #For example, like this

home = Path.home()
output_folder = home / username / "cshl2025" / "output"
output_folder.mkdir(parents=True, exist_ok=True)

# Create the output folder if it doesn't exist
output_folder.mkdir(parents=True, exist_ok=True)

# Save the file
np.save(output_folder / "traces.npy", traces)


tifffile.imwrite(output_folder / "roi_map.tif", roi_map)